# TCM-Classics-Bench · 中医古籍源文本约束型测评基准

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pariskang/TCM-Classics-Bench/blob/claude/tcm-classics-bench-dataset-tu71dt/notebooks/TCM_Classics_Bench.ipynb)

**面向中医古籍大模型的源文本约束型测评基准** — *A Source-Grounded Benchmark for
Large Language Models on Classical Chinese Medicine Texts.*

本 Notebook 在 Colab 中**一键复现全部功能**：

1. 克隆仓库、安装依赖
2. 下载并解压「中醫笈成 / Jicheng-TCM」古籍语料（snapshot `book-20180111`，803 部）
3. 解析编目 → `catalog.jsonl`
4. **简易模式**：数秒生成 5000 道已校验测试题（无需 API key）
5. 源文本约束校验（evidence 回指）
6. 可视化分布（任务 / 书目 / 朝代 / 品质）
7. 浏览样题（T1 句读、T6 方剂结构解析）
8. **NER 子集（T4）**：确定性抽取命名实体（方名/药材/剂量/炮制），F1 评分
9. **LLM 接入与密钥**：Anthropic / Azure / Poe / LiteLLM + 难度（Hard/Expert）
10. **用 LLM 全量生成加强难度试题**（T2/T3/T8/T9/T11，含单选+干扰项）+ 并发 +
    实时写入 Google Drive（断线不丢）+ tqdm 进度条
11. 全量构建 `build_release` + `STATS.json`
12. 下载结果、运行测试

> 运行环境：菜单 *代码执行程序 → 全部运行*。无需 GPU。整段约 1–2 分钟。
> 数据授权见仓库 `DATA_LICENSE.md`（公共领域文本编辑 CC0；站方自创 CC BY 4.0，逐页核查）。

## 1. 环境准备 · Clone & install

In [ ]:
import os

REPO = "https://github.com/pariskang/TCM-Classics-Bench.git"
BRANCH = "claude/tcm-classics-bench-dataset-tu71dt"   # 合并到 main 后改成 "main"

if not os.path.exists("/content/TCM-Classics-Bench"):
    !git clone --branch $BRANCH --depth 1 $REPO /content/TCM-Classics-Bench
%cd /content/TCM-Classics-Bench
!git log --oneline -1

In [ ]:
# 核心依赖：繁简转换 + 纯 Python 7z 解压 + 进度条
!pip install -q opencc py7zr tqdm
# 可选 LLM provider（用到再装；任选其一/多）：
# !pip install -q anthropic        # provider=anthropic
# !pip install -q openai           # provider=azure / poe
# !pip install -q litellm          # provider=litellm
print("deps ready")

## 2. 下载语料 · Download & extract corpus

从 [中醫笈成](https://jicheng.tw/tcm/download.html) 下载 `book-20180111.7z`（约 69 MB）
并用 `py7zr` 解压到 `corpus_src/book/`（每部古籍一个目录）。

In [ ]:
import os, urllib.request, py7zr

URL = "https://jicheng.tw/files/jcw/book-20180111.7z"
ROOT = "corpus_src/book"
os.makedirs(ROOT, exist_ok=True)
archive = "corpus_src/book-20180111.7z"

if not os.path.exists(archive):
    print("downloading ~69MB ...")
    urllib.request.urlretrieve(URL, archive)

if len(os.listdir(ROOT)) <= 1:
    print("extracting...")
    with py7zr.SevenZipFile(archive, "r") as z:
        z.extractall(ROOT)

print("books extracted:", len(os.listdir(ROOT)))

## 3. 解析与编目 · Catalog（全部 803 部书的元数据）

In [ ]:
!python -m tcm_bench catalog --root corpus_src/book --out data/catalog.jsonl

import pandas as pd, json
cat = pd.DataFrame(json.loads(l) for l in open("data/catalog.jsonl"))
print("books:", len(cat))
display(cat[["book_title_trad","author","dynasty","category_level_1",
             "quality_score_source","quality_tier"]].head(10))
print("\n按一级分类:"); display(cat["category_level_1"].value_counts())
print("\n按品质分层:"); display(cat["quality_tier"].value_counts())

## 4. 简易模式 · 5000 道测试题（确定性，无需 API key）

`simple` 一条命令：摄取 pilot 语料 → 生成 T1（句读恢复）+ T6（方剂结构解析）→
**逐题源文本校验** → 按 (任务, 书目) 轮转均衡抽样到 N 道。

In [ ]:
!python -m tcm_bench simple --root corpus_src/book --n 5000 --out test_5k.jsonl

In [ ]:
import pandas as pd, json
items = pd.DataFrame(json.loads(l) for l in open("test_5k.jsonl"))
print("total:", len(items))
print("\n按任务:"); display(items["task_code"].value_counts())
print("\n按书目:"); display(items["book_id"].value_counts())

## 5. 可视化 · 分布图

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
items["task_code"].value_counts().plot.bar(ax=ax[0], title="Items by task", rot=0)
items["book_id"].value_counts().plot.barh(ax=ax[1], title="Items by book")
ax[1].set_yticklabels([])  # CJK 字体在 Colab 默认缺失，标签见上方表格
plt.tight_layout(); plt.show()

## 6. 浏览样题 · Inspect items（T1 句读 / T6 方剂解析）

In [ ]:
import json, textwrap

def show(it):
    print("="*70)
    print("task :", it["task_code"], "|", it["task"], "| difficulty:", it.get("difficulty"))
    print("book :", it["evidence"]["book_title_trad"], "| 底本:", it["evidence"].get("base_text"))
    print("Q    :", it["question"])
    print("context:", textwrap.shorten(str(it["context"]), 160))
    print("answer :", json.dumps(it["answer"], ensure_ascii=False)[:400])
    if it.get("safety_note"): print("safety :", it["safety_note"])

rows = [json.loads(l) for l in open("test_5k.jsonl")]
show(next(r for r in rows if r["task_code"] == "T1"))
show(next(r for r in rows if r["task_code"] == "T6"))

## 7. 源文本约束校验 · Validation

每道题都必须能回指原文：evidence span、方名、药材、剂量都要在原文出现；
`external_required` 题被逐出正式集。`simple` 已内置校验，这里对仓库内置的
`data/bench/` 样本再独立校验一次（对应 `data/corpus/`）。

In [ ]:
!python -m tcm_bench validate --items data/bench/pilot2_zhongjing.jsonl --corpus data/corpus/pilot2_zhongjing.jsonl

## 8. NER 子集 · T4 命名实体识别（确定性，源文本约束）

T4 是从**方剂段落**（`<F>` 块）**确定性**抽取的命名实体识别试题：
方名（`FORMULA`）、药材（`HERB`）、剂量（`DOSE`）、炮制法（`PREP`）。
金标签均**逐字取自原文**，无需 LLM，天然源文本约束。
`build_release` 把 T4 子集单独存入 `data/bench_ner/`，供评测模块以 **F1** 评分。

> **为什么 NER 重要？**
> 方剂解析是中医古籍理解的核心能力。能否准确识别“桂枝 HERB · 三两 DOSE · 去皮 PREP”
> 直接检验模型对古文本词素拆解与语义归类的能力，比生成式 QA 更易量化。

In [ ]:
# T4 NER 子集统计（仓库内置 data/bench_ner/）
import json, collections, pandas as pd, matplotlib.pyplot as plt

ner_items = [json.loads(l) for l in open('data/bench_ner/pilot2_zhongjing.jsonl', encoding='utf-8')]
print(f'T4 NER 子集：{len(ner_items)} 道题（仲景方证 pilot）')

all_entities = [e for it in ner_items for e in it.get('answer', {}).get('entities', [])]
type_counts = collections.Counter(e['type'] for e in all_entities)
print('\n实体类型分布：')
for t, c in type_counts.most_common():
    print(f'  {t:10s}: {c:5d} 个')

book_counts = collections.Counter(it['book_id'] for it in ner_items)
print('\n书目分布：')
for b, c in book_counts.most_common():
    print(f'  {b}: {c} 道')

fig, ax = plt.subplots(figsize=(7, 3))
pd.Series(type_counts).sort_values().plot.barh(ax=ax, title='T4 NER entity-type distribution')
plt.tight_layout(); plt.show()

In [ ]:
# 浏览几条 NER 样题（原文段落 -> 金标签实体）
import json, textwrap

ner_items = [json.loads(l) for l in open('data/bench_ner/pilot2_zhongjing.jsonl', encoding='utf-8')]

for it in ner_items[:3]:
    print('='*70)
    print('book   :', it['evidence']['book_title_trad'])
    print('context:', textwrap.shorten(it['context'], 220))
    print('Q      :', it['question'])
    print('entities:')
    for e in it['answer']['entities']:
        extras = []
        if e.get('dose'):        extras.append(f"剂量={e['dose']}")
        if e.get('preparation'): extras.append(f"炮制={e['preparation']}")
        tag = ('  ' + ' | '.join(extras)) if extras else ''
        print(f"  [{e['type']:8s}] 「{e['text']}」{tag}")
    print('spans  :', it['evidence']['spans'])

## 9. LLM 接入与密钥 · Anthropic / Azure / Poe / LiteLLM

确定性生成器只覆盖 T1 / T6。**更难**的任务（翻译、术语、方证对应、类方鉴别、安全禁忌…）
走 LLM。本节**填入密钥并构建一个全局 `LLM` 生成器**（带难度），供第 9 节直接使用 —
这样第 9 节一定是真正在调用模型，而不会悄悄退回确定性。

In [ ]:
#@title 选择 provider / 模型 / 难度，并填写密钥 { display-mode: "form" }
import getpass, os
PROVIDER = "anthropic"  #@param ["anthropic", "azure", "poe", "litellm"]
MODEL = ""              #@param {type:"string"}
DIFFICULTY = "Hard"     #@param ["Medium", "Hard", "Expert"]
MODEL = MODEL.strip() or None

def setkey(name):
    if not os.environ.get(name):
        os.environ[name] = getpass.getpass(f"{name}: ")

if PROVIDER == "anthropic":
    !pip install -q anthropic
    setkey("ANTHROPIC_API_KEY")
elif PROVIDER == "poe":
    !pip install -q openai
    setkey("POE_API_KEY")
elif PROVIDER == "azure":
    !pip install -q openai
    for k in ["AZURE_OPENAI_API_KEY", "AZURE_OPENAI_ENDPOINT"]:
        setkey(k)
    os.environ.setdefault("AZURE_OPENAI_API_VERSION", "2024-06-01")
elif PROVIDER == "litellm":
    !pip install -q litellm

from tcm_bench.generate import LLMGenerator
from tcm_bench.llm import make_client
LLM = LLMGenerator(make_client(PROVIDER, MODEL), difficulty=DIFFICULTY)
print("LLM ready ->", PROVIDER, "| model:", LLM.model, "| difficulty:", DIFFICULTY)

In [ ]:
# 冒烟测试：在《難經》上真正调用一次 LLM，生成一道高难度 T8（方证对应单选题）
import json
from pathlib import Path
from tcm_bench import ingest
from tcm_bench.validate import validate_item

rec = next(r.to_dict() for r in ingest.ingest_book(Path("corpus_src/book/難經"), "2026-06-26")
           if len(r.to_dict()["raw_text_trad"]) > 60)
item = LLM.generate("T8", rec)          # ← 实际 API 调用
d = item.to_dict()
print("difficulty:", d["difficulty"], "| inference:", d["inference_level"])
print("Q:", d["question"])
for o in d.get("options", []): print("  ", o)
print("answer:", d["answer"])
print("valid:", validate_item(d, rec["raw_text_trad"]).ok)

你也可以直接用命令行（`--difficulty` 难度、`--workers` 并发、`--progress` 进度条）：

```bash
# Poe，专家级、16 路并发 + 进度条（注意 --tasks 是需要 LLM 的任务）
export POE_API_KEY=...
python -m tcm_bench simple --root corpus_src/book --n 300 --tasks T2 T3 T8 T9 T11 \
       --llm --provider poe --model Claude-Sonnet-4 \
       --difficulty Expert --workers 16 --progress

# Azure OpenAI
export AZURE_OPENAI_API_KEY=...  AZURE_OPENAI_ENDPOINT=https://<res>.openai.azure.com
python -m tcm_bench generate --corpus data/corpus/pilot1_classics.jsonl \
       --out llm_items.jsonl --tasks T2 T3 T8 --llm --provider azure \
       --model <deployment> --difficulty Hard --workers 8 --progress
```

## 10. 用 LLM 全量生成（加强难度）+ 实时写入 Google Drive + 进度条

**本节真正调用 LLM。** 任务默认是需要推理的难任务（T2/T3/T8/T9/T11），用第 8 节构建的
`LLM`（含难度）+ 线程池并发生成，每题**逐条 flush 到 Google Drive**（断线不丢），
`tqdm` 显示进度。先挂载 Drive：

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive")
OUT_DIR = "/content/drive/MyDrive/tcm_classics_bench"
os.makedirs(OUT_DIR, exist_ok=True)
print("output dir:", OUT_DIR)

> **断点续跑（Resume）**：本 cell 默认 `RESUME=True`。它先读取已写入 Drive 的题目，
> 以 `(passage_id, task_code)` 为唯一键**跳过已完成的**，并以 `"a"` 追加模式写入 ——
> 因此把 `N` 从 5000 改成 33000 后**重跑会从上次的后面继续**，不会重头来过、也不会覆盖。
> 设 `RESUME=False` 则从头开始（旧文件自动备份为 `.bak`）。
>
> 并发已是**有界滑动窗口**（约 `2×WORKERS` 在途），中途达到 `N` 即停，不会多花 API。
> `WORKERS` 不宜过大（建议 32–64），过高易触发 provider 限流。

In [ ]:
#@title LLM 全量生成（加强难度 / 断点续跑）+ 实时写入 Drive + 进度条 { display-mode: "form" }
N = 33000                      #@param {type:"integer"}
LLM_TASKS = "T2 T3 T8 T9 T11"  #@param {type:"string"}
WORKERS = 48                   #@param {type:"integer"}
RESUME = True                  #@param {type:"boolean"}

import os, json, random
from pathlib import Path
from tqdm.auto import tqdm
from tcm_bench import ingest, taxonomy
from tcm_bench.generate import generate_items_concurrent
from tcm_bench.validate import validate_item

assert "LLM" in globals(), "请先运行第 8 节构建 LLM（填密钥）。"
tasks = LLM_TASKS.split()
assert any(t not in {"T1", "T6"} for t in tasks), \
    "LLM_TASKS 必须包含需要 LLM 的任务（如 T2/T3/T8/T9/T11），否则不会调用模型。"

# 摄取 pilot 语料并打散（确定性 seed -> job 顺序可复现，续跑稳定）
records = []
for pilot in taxonomy.PILOT_CORPORA:
    for name in taxonomy.PILOT_CORPORA[pilot]:
        d = Path("corpus_src/book") / name
        if (d / "index.txt").exists():
            records += [r.to_dict() for r in ingest.ingest_book(d, "2026-06-26", min_chars=40)]
random.Random(20260626).shuffle(records)
src = {r["passage_id"]: r["raw_text_trad"] for r in records}

out_path = os.path.join(OUT_DIR, "tcm_llm_questions.jsonl")

# 1) 读取已完成（断点续跑）：唯一键 (passage_id, task_code)
done, n_ok = set(), 0
if RESUME and os.path.exists(out_path):
    with open(out_path, encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            try:
                o = json.loads(line)            # 跳过断电写一半的残行
            except json.JSONDecodeError:
                continue
            done.add((o["passage_id"], o["task_code"])); n_ok += 1
    print(f"续跑：已有 {n_ok} 道，跳过 {len(done)} 个已完成 (passage, task)")
elif (not RESUME) and os.path.exists(out_path):
    bak = out_path + ".bak"; os.replace(out_path, bak)
    print(f"从头开始：旧文件已备份为 {bak}")

if n_ok >= N:
    print(f"已达到目标 {N} 道，无需继续。")
else:
    # 2) 跳过已完成 -> 接着生成；"a" 追加写入；进度条从已完成处起步
    #    stats 记录 LLM 侧损耗；reasons/inf 记录校验淘汰的原因，便于定位 5000->? 的缺口
    import collections
    stats = {}
    n_bad = 0
    reasons = collections.Counter()
    inf_bad = collections.Counter()
    gen = generate_items_concurrent(records, tasks, llm=LLM,
                                    max_workers=WORKERS, skip=done, stats=stats)
    with open(out_path, "a", encoding="utf-8") as fh, \
         tqdm(total=N, initial=n_ok, unit="题", desc=f"LLM/{LLM.difficulty}") as bar:
        for it in gen:
            d = it.to_dict()
            res = validate_item(d, src.get(d["passage_id"], ""))
            if res.ok:
                fh.write(json.dumps(d, ensure_ascii=False) + "\n"); fh.flush()  # 实时落盘
                n_ok += 1; bar.update(1)
            else:
                n_bad += 1
                inf_bad[d.get("inference_level")] += 1
                reasons[res.errors[0] if res.errors else "unknown"] += 1
            if n_ok >= N:
                break

    # 3) 诊断报告：把 N 与实际产出之间的缺口拆清楚
    print(f"\n完成：累计 {n_ok} 道已写入 {out_path}")
    print("—— 本次诊断 ——")
    print(f"  目标 N           : {N}")
    print(f"  LLM 任务总数      : {stats.get('jobs_total')}（= 待生成 job）")
    print(f"  模型产出(已解析)   : {stats.get('yielded')}")
    print(f"  API 报错丢弃       : {stats.get('errored')}  (限流/超时等；可调小 WORKERS 或重跑续生成)")
    print(f"  JSON 解析失败      : {stats.get('parse_failed')}")
    print(f"  校验通过(本次写入) : {n_ok - len(done)}")
    print(f"  校验淘汰          : {n_bad}")
    if reasons:
        print("  淘汰原因 Top5     :")
        for r, c in reasons.most_common(5):
            print(f"     {c:5d}  {r}")
        print(f"  其中 external_required 被剔除: {inf_bad.get('external_required', 0)}")
    if n_ok < N:
        miss = N - n_ok
        print(f"\n⚠️ 比目标少 {miss} 道。常见原因：")
        print("   • API 报错丢弃多 → 调小 WORKERS（如 16~32）后重跑本 cell（会自动续生成）；")
        print("   • 校验淘汰多 → 见上方 Top5；external_required 多说明题目超出本段原文，")
        print("     可在第 8 节把 DIFFICULTY 调成 Hard（而非 Expert），或增加 LLM_TASKS（如加 T2 T3）；")
        print("   • 任务池不足 → 增加 LLM_TASKS，或把 min_chars 调小以纳入更多段落。")

In [ ]:
# 看一道刚生成的 LLM 难题（含选项与干扰项排除理由）
import json
rows = [json.loads(l) for l in open(os.path.join(OUT_DIR, "tcm_llm_questions.jsonl"))]
mcq = next((r for r in rows if r.get("options")), rows[0])
print("task:", mcq["task_code"], "| difficulty:", mcq["difficulty"])
print("Q:", mcq["question"])
for o in mcq.get("options", []): print("  ", o)
print("answer:", mcq.get("answer"))
for dz in mcq.get("distractors", []):
    print("  排除", dz.get("option"), "->", dz.get("exclusion_reason"))
print("evidence:", mcq["evidence"]["spans"])

## 11. 全量构建 · build_release（catalog + 三个 pilot + bench + STATS）

In [ ]:
!python scripts/build_release.py --root corpus_src/book
import json
print(json.dumps(json.load(open("data/STATS.json")), ensure_ascii=False, indent=2)[:1500])

## 12. 下载结果 · Download

In [ ]:
from google.colab import files
files.download("test_5k.jsonl")        # 5000 道题
# files.download("data/catalog.jsonl") # 803 部书元数据

## 13. 单元测试 · Tests

In [ ]:
!python -m pytest -q

---

仓库：**pariskang/TCM-Classics-Bench**（分支 `claude/tcm-classics-bench-dataset-tu71dt`）
· 协议见 `PROTOCOL.md` · 授权见 `DATA_LICENSE.md`
· 题目仅作古籍理解测评，**不构成任何用药建议**。